# Session 4: Multi-Agent Workflows & Agentic Systems (Instructor Notebook -- Full Solutions)

This is the **instructor version** of Session 4. It contains the same demos as the student notebook, plus **complete, working solutions** for all four tasks.

## Learning Objectives

By the end of this session, students will be able to:

1. **Design multi-agent architectures** using supervisor-worker patterns
2. **Implement agent handoffs** with context preservation
3. **Run agents in parallel** and merge their results
4. **Build collaborative agent teams** for complex tasks
5. **Integrate all Day 2 concepts** into an end-to-end system
6. **Evaluate trade-offs** between single-agent and multi-agent designs

In [ ]:
from langgraph.graph import StateGraph, END
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage, AIMessage, SystemMessage
from typing import TypedDict, Literal
import json
import os

print("All imports successful!")

---
## Demos (Follow Along)

Demos are identical to the student notebook.

### Demo 1: Supervisor-Worker Pattern

In [ ]:
# Demo 1 - Supervisor-Worker Pattern

class SupervisorState(TypedDict):
    task: str
    assigned_to: str
    worker_output: str
    final_response: str

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

def supervisor_node(state):
    r = llm.invoke([SystemMessage(content="Assign to: 'researcher', 'writer', or 'coder'. One word."), HumanMessage(content=state["task"])])
    assigned = r.content.strip().lower()
    print(f"  Supervisor -> {assigned}")
    return {"assigned_to": assigned}

def researcher_node(state):
    r = llm.invoke([SystemMessage(content="You are a research analyst."), HumanMessage(content=state["task"])])
    return {"worker_output": f"[Research] {r.content}"}

def writer_node(state):
    r = llm.invoke([SystemMessage(content="You are a creative writer."), HumanMessage(content=state["task"])])
    return {"worker_output": f"[Writer] {r.content}"}

def coder_node(state):
    r = llm.invoke([SystemMessage(content="You are an expert programmer."), HumanMessage(content=state["task"])])
    return {"worker_output": f"[Coder] {r.content}"}

def review_node(state):
    r = llm.invoke([SystemMessage(content="Review and finalize."), HumanMessage(content=f"Task: {state['task']}\nOutput: {state['worker_output']}" )])
    return {"final_response": r.content}

def route_worker(state):
    a = state["assigned_to"]
    if "researcher" in a: return "researcher"
    if "writer" in a: return "writer"
    return "coder"

graph = StateGraph(SupervisorState)
graph.add_node("supervisor", supervisor_node)
graph.add_node("researcher", researcher_node)
graph.add_node("writer", writer_node)
graph.add_node("coder", coder_node)
graph.add_node("review", review_node)
graph.set_entry_point("supervisor")
graph.add_conditional_edges("supervisor", route_worker, {"researcher": "researcher", "writer": "writer", "coder": "coder"})
for n in ["researcher", "writer", "coder"]: graph.add_edge(n, "review")
graph.add_edge("review", END)
app = graph.compile()

r = app.invoke({"task": "Explain the pros and cons of microservices", "assigned_to": "", "worker_output": "", "final_response": ""})
print(f"Response: {r['final_response'][:200]}...")

### Demo 2: Agent-to-Agent Handoff

In [ ]:
# Demo 2 - Agent Handoff

class HandoffState(TypedDict):
    user_request: str
    draft: str
    review_notes: str
    final_output: str
    handoff_context: str

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0.5)

def drafting_agent(state):
    r = llm.invoke([SystemMessage(content="Create a draft."), HumanMessage(content=state["user_request"])])
    return {"draft": r.content, "handoff_context": "Draft done. Review for accuracy."}

def review_agent(state):
    r = llm.invoke([SystemMessage(content="Review and give notes."), HumanMessage(content=f"Context: {state['handoff_context']}\nDraft: {state['draft']}" )])
    return {"review_notes": r.content}

def editing_agent(state):
    r = llm.invoke([SystemMessage(content="Apply notes and finalize."), HumanMessage(content=f"Draft: {state['draft']}\nNotes: {state['review_notes']}" )])
    return {"final_output": r.content}

graph = StateGraph(HandoffState)
graph.add_node("drafter", drafting_agent)
graph.add_node("reviewer", review_agent)
graph.add_node("editor", editing_agent)
graph.set_entry_point("drafter")
graph.add_edge("drafter", "reviewer")
graph.add_edge("reviewer", "editor")
graph.add_edge("editor", END)
app = graph.compile()

r = app.invoke({"user_request": "Explain how neural networks learn", "draft": "", "review_notes": "", "final_output": "", "handoff_context": ""})
print(f"Final: {r['final_output'][:200]}...")

### Demo 3: Parallel Agent Execution

In [ ]:
# Demo 3 - Parallel Agents (sequential simulation)

class ParallelState(TypedDict):
    topic: str
    pros_analysis: str
    cons_analysis: str
    examples_analysis: str
    merged_report: str

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0.5)

def pros_agent(s): return {"pros_analysis": llm.invoke([SystemMessage(content="List 3 advantages."), HumanMessage(content=s["topic"])]).content}
def cons_agent(s): return {"cons_analysis": llm.invoke([SystemMessage(content="List 3 disadvantages."), HumanMessage(content=s["topic"])]).content}
def examples_agent(s): return {"examples_analysis": llm.invoke([SystemMessage(content="Give 3 examples."), HumanMessage(content=s["topic"])]).content}
def merge_agent(s):
    r = llm.invoke([SystemMessage(content="Combine into a balanced report."), HumanMessage(content=f"Pros:{s['pros_analysis']}\nCons:{s['cons_analysis']}\nExamples:{s['examples_analysis']}")])
    return {"merged_report": r.content}

graph = StateGraph(ParallelState)
for name, fn in [("pros", pros_agent), ("cons", cons_agent), ("examples", examples_agent), ("merge", merge_agent)]:
    graph.add_node(name, fn)
graph.set_entry_point("pros")
graph.add_edge("pros", "cons")
graph.add_edge("cons", "examples")
graph.add_edge("examples", "merge")
graph.add_edge("merge", END)
app = graph.compile()

r = app.invoke({"topic": "Using LLMs for code review", "pros_analysis": "", "cons_analysis": "", "examples_analysis": "", "merged_report": ""})
print(f"Report: {r['merged_report'][:300]}...")

### Demo 4: Collaborative Writing

In [ ]:
# Demo 4 - Collaborative Writing

class CollabState(TypedDict):
    topic: str
    outline: str
    draft: str
    polished: str

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0.7)

def outliner(s): return {"outline": llm.invoke([SystemMessage(content="Create a 3-section outline."), HumanMessage(content=s["topic"])]).content}
def writer(s): return {"draft": llm.invoke([SystemMessage(content="Write concise text from outline."), HumanMessage(content=f"{s['topic']}\n{s['outline']}")]).content}
def stylist(s): return {"polished": llm.invoke([SystemMessage(content="Polish for clarity and engagement."), HumanMessage(content=s["draft"])]).content}

graph = StateGraph(CollabState)
graph.add_node("outliner", outliner)
graph.add_node("writer", writer)
graph.add_node("stylist", stylist)
graph.set_entry_point("outliner")
graph.add_edge("outliner", "writer")
graph.add_edge("writer", "stylist")
graph.add_edge("stylist", END)
app = graph.compile()

r = app.invoke({"topic": "Getting started with LangGraph", "outline": "", "draft": "", "polished": ""})
print(f"Article: {r['polished'][:300]}...")

### Demo 5: End-to-End Multi-Agent System

In [ ]:
# Demo 5 - End-to-End System

class E2EState(TypedDict):
    user_input: str
    intent: str
    agent_output: str
    quality_score: str
    final_output: str

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

def classify(s): return {"intent": llm.invoke([SystemMessage(content="Classify: 'question', 'task', or 'chat'. One word."), HumanMessage(content=s["user_input"])]).content.strip().lower()}
def question_agent(s): return {"agent_output": llm.invoke([SystemMessage(content="Answer accurately."), HumanMessage(content=s["user_input"])]).content}
def task_agent(s): return {"agent_output": llm.invoke([SystemMessage(content="Complete step by step."), HumanMessage(content=s["user_input"])]).content}
def chat_agent(s): return {"agent_output": llm.invoke([SystemMessage(content="Be friendly."), HumanMessage(content=s["user_input"])]).content}
def quality(s):
    r = llm.invoke([SystemMessage(content="Rate 1-10 as JSON."), HumanMessage(content=f"Q: {s['user_input']}\nA: {s['agent_output']}")])
    return {"quality_score": r.content, "final_output": s["agent_output"]}

def route(s):
    if "question" in s["intent"]: return "question"
    if "task" in s["intent"]: return "task"
    return "chat"

graph = StateGraph(E2EState)
for name, fn in [("classify", classify), ("question", question_agent), ("task", task_agent), ("chat", chat_agent), ("quality", quality)]:
    graph.add_node(name, fn)
graph.set_entry_point("classify")
graph.add_conditional_edges("classify", route, {"question": "question", "task": "task", "chat": "chat"})
for n in ["question", "task", "chat"]: graph.add_edge(n, "quality")
graph.add_edge("quality", END)
app = graph.compile()

r = app.invoke({"user_input": "What is the capital of Japan?", "intent": "", "agent_output": "", "quality_score": "", "final_output": ""})
print(f"Answer: {r['final_output'][:150]}")

---
## Tasks -- Full Solutions

### Task 1: Build a Two-Agent Supervisor-Worker System

**Approach:** The supervisor classifies tasks as analyst or creative. Conditional edges route to the right worker. A review node checks the output quality.

In [ ]:
# Task 1 - SOLUTION: Two-Agent Supervisor-Worker System

class Task1State(TypedDict):
    task: str
    assigned_to: str
    worker_output: str
    final_response: str

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

def t1_supervisor(state):
    r = llm.invoke([SystemMessage(content="Assign to 'analyst' or 'creative'. One word."), HumanMessage(content=state["task"])])
    assigned = r.content.strip().lower()
    print(f"  Assigned: {assigned}")
    return {"assigned_to": assigned}

def t1_analyst(state):
    r = llm.invoke([SystemMessage(content="You are a data analyst. Provide structured, factual analysis."), HumanMessage(content=state["task"])])
    return {"worker_output": r.content}

def t1_creative(state):
    r = llm.invoke([SystemMessage(content="You are a creative content specialist. Write engaging, compelling content."), HumanMessage(content=state["task"])])
    return {"worker_output": r.content}

def t1_review(state):
    r = llm.invoke([SystemMessage(content="Review this work. Add a brief assessment and finalize."), HumanMessage(content=f"Task: {state['task']}\nOutput: {state['worker_output']}")])
    return {"final_response": r.content}

def t1_route(state):
    return "analyst" if "analyst" in state["assigned_to"] else "creative"

graph = StateGraph(Task1State)
graph.add_node("supervisor", t1_supervisor)
graph.add_node("analyst", t1_analyst)
graph.add_node("creative", t1_creative)
graph.add_node("review", t1_review)
graph.set_entry_point("supervisor")
graph.add_conditional_edges("supervisor", t1_route, {"analyst": "analyst", "creative": "creative"})
graph.add_edge("analyst", "review")
graph.add_edge("creative", "review")
graph.add_edge("review", END)
app = graph.compile()

for task in ["Analyze the trend of AI adoption in healthcare", "Write a tagline for a new AI startup"]:
    r = app.invoke({"task": task, "assigned_to": "", "worker_output": "", "final_response": ""})
    print(f"\nTask: {task}")
    print(f"Response: {r['final_response'][:200]}...\n")

### Task 2: Implement Agent Handoff with Context Preservation

**Approach:** Each agent reads the accumulated handoff context, does its work, and appends its own notes to the context before passing it along. This ensures downstream agents have full situational awareness.

In [ ]:
# Task 2 - SOLUTION: Agent Handoff with Context Preservation

class Task2State(TypedDict):
    topic: str
    research: str
    analysis: str
    presentation: str
    handoff_context: str

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0.5)

def t2_researcher(state):
    r = llm.invoke([SystemMessage(content="Research this topic. Provide key facts and data."), HumanMessage(content=state["topic"])])
    ctx = f"[Researcher] Gathered facts about {state['topic']}. Key areas covered: trends, statistics, key players."
    print(f"  [Researcher] Done")
    return {"research": r.content, "handoff_context": ctx}

def t2_analyst(state):
    r = llm.invoke([
        SystemMessage(content="Analyze this research. Identify key insights and implications."),
        HumanMessage(content=f"Handoff: {state['handoff_context']}\n\nResearch:\n{state['research']}")
    ])
    ctx = state["handoff_context"] + f"\n[Analyst] Identified 3 key insights. Focus presentation on implications."
    print(f"  [Analyst] Done")
    return {"analysis": r.content, "handoff_context": ctx}

def t2_presenter(state):
    r = llm.invoke([
        SystemMessage(content="Create a concise presentation summary with key takeaways."),
        HumanMessage(content=f"Handoff: {state['handoff_context']}\n\nAnalysis:\n{state['analysis']}")
    ])
    print(f"  [Presenter] Done")
    return {"presentation": r.content}

graph = StateGraph(Task2State)
graph.add_node("researcher", t2_researcher)
graph.add_node("analyst", t2_analyst)
graph.add_node("presenter", t2_presenter)
graph.set_entry_point("researcher")
graph.add_edge("researcher", "analyst")
graph.add_edge("analyst", "presenter")
graph.add_edge("presenter", END)
app = graph.compile()

r = app.invoke({"topic": "The future of remote work", "research": "", "analysis": "", "presentation": "", "handoff_context": ""})
print(f"\nPresentation:\n{r['presentation'][:300]}...")
print(f"\nHandoff trail:\n{r['handoff_context']}")

### Task 3: Create a Parallel Research and Synthesis Pipeline

**Approach:** Three research agents (technical, business, social) each investigate a different dimension. The synthesis agent receives all three outputs and creates a unified report.

In [ ]:
# Task 3 - SOLUTION: Parallel Research and Synthesis Pipeline

class Task3State(TypedDict):
    topic: str
    technical_output: str
    business_output: str
    social_output: str
    report: str

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0.5)

def t3_technical(state):
    r = llm.invoke([SystemMessage(content="Research the TECHNICAL aspects: capabilities, limitations, architecture."), HumanMessage(content=state["topic"])])
    print("  [Technical] Done")
    return {"technical_output": r.content}

def t3_business(state):
    r = llm.invoke([SystemMessage(content="Research the BUSINESS aspects: market size, ROI, adoption."), HumanMessage(content=state["topic"])])
    print("  [Business] Done")
    return {"business_output": r.content}

def t3_social(state):
    r = llm.invoke([SystemMessage(content="Research the SOCIAL aspects: impact on jobs, ethics, accessibility."), HumanMessage(content=state["topic"])])
    print("  [Social] Done")
    return {"social_output": r.content}

def t3_synthesis(state):
    r = llm.invoke([
        SystemMessage(content="Synthesize these three perspectives into a comprehensive, balanced report."),
        HumanMessage(content=f"Topic: {state['topic']}\n\nTechnical:\n{state['technical_output']}\n\nBusiness:\n{state['business_output']}\n\nSocial:\n{state['social_output']}")
    ])
    return {"report": r.content}

graph = StateGraph(Task3State)
graph.add_node("technical", t3_technical)
graph.add_node("business", t3_business)
graph.add_node("social", t3_social)
graph.add_node("synthesis", t3_synthesis)
graph.set_entry_point("technical")
graph.add_edge("technical", "business")
graph.add_edge("business", "social")
graph.add_edge("social", "synthesis")
graph.add_edge("synthesis", END)
app = graph.compile()

r = app.invoke({"topic": "Generative AI in education", "technical_output": "", "business_output": "", "social_output": "", "report": ""})
print(f"\nReport:\n{r['report'][:500]}...")

### Task 4: Design and Document a Complete Multi-Agent Architecture

**Approach:** We build a code review pipeline with three specialized agents (logic reviewer, security reviewer, style reviewer) and a supervisor that produces a final report. This demonstrates conditional routing + fan-out + synthesis.

In [ ]:
# Task 4 - SOLUTION: Complete Multi-Agent Code Review System

print("""Architecture: Code Review Multi-Agent System
Agents:
  1. Triage Agent - determines review priority and type
  2. Logic Reviewer - checks correctness and edge cases
  3. Security Reviewer - identifies vulnerabilities
  4. Style Reviewer - checks code style and readability
  5. Report Agent - synthesizes all reviews into final report

Flow: triage -> logic -> security -> style -> report -> END
      (triage can skip security for non-critical code)
""")

class CodeReviewState(TypedDict):
    code: str
    priority: str
    logic_review: str
    security_review: str
    style_review: str
    final_report: str

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

def triage_agent(state):
    r = llm.invoke([SystemMessage(content="Classify code priority: 'critical' (needs security review) or 'standard'. One word."), HumanMessage(content=state["code"])])
    priority = r.content.strip().lower()
    print(f"  [Triage] Priority: {priority}")
    return {"priority": priority}

def logic_reviewer(state):
    r = llm.invoke([SystemMessage(content="Review code for logical correctness, edge cases, and bugs."), HumanMessage(content=state["code"])])
    return {"logic_review": r.content}

def security_reviewer(state):
    r = llm.invoke([SystemMessage(content="Review code for security vulnerabilities: injection, auth issues, data exposure."), HumanMessage(content=state["code"])])
    return {"security_review": r.content}

def style_reviewer(state):
    r = llm.invoke([SystemMessage(content="Review code style: naming, structure, readability, best practices."), HumanMessage(content=state["code"])])
    return {"style_review": r.content}

def report_agent(state):
    reviews = f"Logic: {state['logic_review']}\n\nSecurity: {state['security_review'] or 'Skipped'}\n\nStyle: {state['style_review']}"
    r = llm.invoke([SystemMessage(content="Create a final code review report with an overall verdict."), HumanMessage(content=f"Code:\n{state['code']}\n\nReviews:\n{reviews}")])
    return {"final_report": r.content}

def route_after_logic(state):
    return "security" if "critical" in state["priority"] else "style"

graph = StateGraph(CodeReviewState)
graph.add_node("triage", triage_agent)
graph.add_node("logic", logic_reviewer)
graph.add_node("security", security_reviewer)
graph.add_node("style", style_reviewer)
graph.add_node("report", report_agent)
graph.set_entry_point("triage")
graph.add_edge("triage", "logic")
graph.add_conditional_edges("logic", route_after_logic, {"security": "security", "style": "style"})
graph.add_edge("security", "style")
graph.add_edge("style", "report")
graph.add_edge("report", END)
app = graph.compile()

# Test with sample code
sample_code = '''def login(username, password):
    query = f"SELECT * FROM users WHERE name='{username}' AND pass='{password}'"
    result = db.execute(query)
    if result:
        return True
    return False'''

r = app.invoke({"code": sample_code, "priority": "", "logic_review": "", "security_review": "", "style_review": "", "final_report": ""})
print(f"\nCode Review Report:\n{r['final_report']}")

---
## Summary

In this capstone session students built multi-agent systems:

1. **Supervisor-Worker** -- Central routing to specialized workers.
2. **Agent Handoffs** -- Structured context passing between agents.
3. **Parallel Execution** -- Fan-out/fan-in for independent subtasks.
4. **Collaborative Writing** -- Multiple agents contributing to shared output.
5. **End-to-End Systems** -- Complete architectures with classification, specialization, and quality review.

**Instructor Notes:**
- Emphasize that multi-agent is not always better -- use the simplest architecture that works.
- For Task 4, encourage students to think about failure modes: what if an agent produces bad output?
- Discuss real-world considerations: cost (more agents = more API calls), latency, and debugging complexity.
- Preview Day 3: production deployment, monitoring, and advanced patterns.

**Day 2 Complete!**